In [1]:
import time
from pathlib import Path

import torch
import pandas as pd

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

In [2]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("Device:", device)


if torch.cuda.is_available():
    print(
        torch.cuda.get_device_name(0)
    )

Device: cuda
NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
MODEL_NAME = "alirezamsh/small100"


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)


model.to(device)


model.eval()


print("Model loaded")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\WingYouther\.cache\huggingface\hub\models--alirezamsh--small100. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 2.42MB            

...

special_tokens_map.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/275 [00:00<?, ?it/s]

Model loaded


In [4]:
print(
    tokenizer.lang_code_to_token.keys()
)

dict_keys(['af', 'am', 'ar', 'ast', 'az', 'ba', 'be', 'bg', 'bn', 'br', 'bs', 'ca', 'ceb', 'cs', 'cy', 'da', 'de', 'el', 'en', 'es', 'et', 'fa', 'ff', 'fi', 'fr', 'fy', 'ga', 'gd', 'gl', 'gu', 'ha', 'he', 'hi', 'hr', 'ht', 'hu', 'hy', 'id', 'ig', 'ilo', 'is', 'it', 'ja', 'jv', 'ka', 'kk', 'km', 'kn', 'ko', 'lb', 'lg', 'ln', 'lo', 'lt', 'lv', 'mg', 'mk', 'ml', 'mn', 'mr', 'ms', 'my', 'ne', 'nl', 'no', 'ns', 'oc', 'or', 'pa', 'pl', 'ps', 'pt', 'ro', 'ru', 'sd', 'si', 'sk', 'sl', 'so', 'sq', 'sr', 'ss', 'su', 'sv', 'sw', 'ta', 'th', 'tl', 'tn', 'tr', 'uk', 'ur', 'uz', 'vi', 'wo', 'xh', 'yi', 'yo', 'zh', 'zu'])


In [5]:
def translate(
    text,
    src_lang,
    tgt_lang
):

    tokenizer.src_lang = src_lang


    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)



    with torch.no_grad():

        generated = model.generate(
            **inputs,

            forced_bos_token_id=
            tokenizer.get_lang_id(
                tgt_lang
            ),

            max_length=128
        )


    result = tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )


    return result

In [6]:
text = "I am a student."


result = translate(
    text,
    "en",
    "uz"
)


print(result)

: I am a student.


In [7]:
text = "Men talabaman."


result = translate(
    text,
    "uz",
    "en"
)


print(result)

bilan qilmaydi.


In [8]:
test_cases = [

    {
        "src_lang":"zh",
        "tgt_lang":"en",
        "text":"我明天早上八点去机场。"
    },


    {
        "src_lang":"en",
        "tgt_lang":"zh",
        "text":"I will go to the airport tomorrow morning."
    },


    {
        "src_lang":"ru",
        "tgt_lang":"en",
        "text":"Я завтра утром поеду в аэропорт."
    },


    {
        "src_lang":"en",
        "tgt_lang":"ru",
        "text":"I will go to the airport tomorrow morning."
    },


    {
        "src_lang":"uz",
        "tgt_lang":"en",
        "text":"Men ertaga ertalab aeroportga boraman."
    },


    {
        "src_lang":"en",
        "tgt_lang":"uz",
        "text":"I will go to the airport tomorrow morning."
    }

]

In [9]:
results=[]


for item in test_cases:


    start=time.time()


    output=translate(
        item["text"],
        item["src_lang"],
        item["tgt_lang"]
    )


    latency=time.time()-start



    results.append(
        {
            "src_lang":
                item["src_lang"],

            "tgt_lang":
                item["tgt_lang"],

            "source":
                item["text"],

            "translation":
                output,

            "latency":
                latency
        }
    )

In [11]:
df=pd.DataFrame(results)


df

,src_lang,tgt_lang,source,translation,latency
0,zh,en,我明天早上八点去机场。,明天早上8点去机场。,0.167167
1,en,zh,I will go to the airport tomorrow morning.,le to the airport tomorrow morning.,0.065746
2,ru,en,Я завтра утром поеду в аэропорт.,жусь завтра утром в аэропорт.,0.085106
3,en,ru,I will go to the airport tomorrow morning.,ing to the airport tomorrow morning.,0.077157
4,uz,en,Men ertaga ertalab aeroportga boraman.,idagi aeroportda boraman.,0.071815
5,en,uz,I will go to the airport tomorrow morning.,to the airport tomorrow morning.,0.080673


In [12]:
df["latency"].describe()

count    6.000000
mean     0.091277
std      0.037788
min      0.065746
25%      0.073150
50%      0.078915
75%      0.083998
max      0.167167
Name: latency, dtype: float64

In [13]:
OUTPUT_DIR = Path(
    "./results"
)


OUTPUT_DIR.mkdir(
    exist_ok=True
)


df.to_csv(
    OUTPUT_DIR /
    "small100_baseline.csv",
    index=False,
    encoding="utf-8-sig"
)

In [15]:
test_cases = [

    {
        "src_lang":"en",
        "tgt_lang":"uz",
        "text":"I am a student."
    },

    {
        "src_lang":"uz",
        "tgt_lang":"en",
        "text":"Men talabaman."
    },

    {
        "src_lang":"en",
        "tgt_lang":"uz",
        "text":"I will go to the airport tomorrow morning."
    },

    {
        "src_lang":"uz",
        "tgt_lang":"en",
        "text":"Men ertaga ertalab aeroportga boraman."
    },


    {
        "src_lang":"zh",
        "tgt_lang":"en",
        "text":"我明天早上八点去机场。"
    },


    {
        "src_lang":"ru",
        "tgt_lang":"en",
        "text":"Я завтра утром поеду в аэропорт."
    }

]

In [16]:
results=[]


for item in test_cases:

    output=translate(
        item["text"],
        item["src_lang"],
        item["tgt_lang"]
    )

    results.append(
        {
            "src_lang":item["src_lang"],
            "tgt_lang":item["tgt_lang"],
            "source":item["text"],
            "translation":output
        }
    )


pd.DataFrame(results)

,src_lang,tgt_lang,source,translation
0,en,uz,I am a student.,: I am a student.
1,uz,en,Men talabaman.,bilan qilmaydi.
2,en,uz,I will go to the airport tomorrow morning.,to the airport tomorrow morning.
3,uz,en,Men ertaga ertalab aeroportga boraman.,idagi aeroportda boraman.
4,zh,en,我明天早上八点去机场。,明天早上8点去机场。
5,ru,en,Я завтра утром поеду в аэропорт.,жусь завтра утром в аэропорт.


In [17]:
import pandas as pd


df = pd.read_csv(
    r"D:\dev\projects\fourlang_translation\data\clean\en_uz\tatoeba_en_uz_latin.csv"
)


df.head()

,en_id,en,uz_id,uz,uz_has_cyrillic
0,16996,Mind your own business!,423846,Ishingizni qiling!,False
1,16996,Mind your own business!,423847,Ishingni qil!,False
2,16996,Mind your own business!,423848,Ishinglarni qilinglar!,False
3,19904,The customer did not come.,2790055,Xaridor kelmadi.,False
4,20392,Never mind.,423857,Hech gap yo'q.,False


In [18]:
test_df=df.sample(
    20,
    random_state=42
)


test_df

,en_id,en,uz_id,uz,uz_has_cyrillic
421,2362724,Good night.,13758141,Xayrli tun.,False
75,2499490,Where is Laurie?,2499491,Laurie qayerda?,False
177,64956,I was hungry.,3834516,Men och edim.,False
30,410907,Where are my watches?,423842,Mening soatlarim qani?,False
360,5293174,"I refuse to use public restrooms, as they are ...",11282118,Juda antigigiyenik bo‘lganidan jamoat hojatxon...,False
272,4851580,Where are the books?,4866623,Kitoblar qayerda?,False
155,1866502,I live in a rural area.,3764310,Qishloq joyida yashayman.,False
152,1761849,Do you drink beer?,3764115,Pivo ichasizmi?,False
165,2739964,I have a friend.,3776251,Mening bir do'stim bor.,False
175,16908,Your book is on the desk.,3834134,Kitobingiz stol ustida.,False


In [19]:
for _,row in test_df.iterrows():

    pred=translate(
        row["en"],
        "en",
        "uz"
    )

    print(
        "原文:",
        row["en"]
    )

    print(
        "参考:",
        row["uz"]
    )

    print(
        "预测:",
        pred
    )

    print("-"*50)

原文: Good night.
参考: Xayrli tun.
预测: al good night.
--------------------------------------------------
原文: Where is Laurie?
参考: Laurie qayerda?
预测: : Where is Laurie?
--------------------------------------------------
原文: I was hungry.
参考: Men och edim.
预测: was hungry.
--------------------------------------------------
原文: Where are my watches?
参考: Mening soatlarim qani?
预测: : Where are my watches?
--------------------------------------------------
原文: I refuse to use public restrooms, as they are very unhygienic.
参考: Juda antigigiyenik bo‘lganidan jamoat hojatxonalaridan foydalanishni rad etaman.
预测: to use public restrooms, as they are very unhygienic.
--------------------------------------------------
原文: Where are the books?
参考: Kitoblar qayerda?
预测: : Where are the books?
--------------------------------------------------
原文: I live in a rural area.
参考: Qishloq joyida yashayman.
预测: al: I live in a rural area.
--------------------------------------------------
原文: Do you drink beer